In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "async-geotiff>=0.4",
#     "lonboard>=0.15.0",
#     "numpy>=2",
#     "obstore>=0.9.2",
#     "pillow>=12.1.1",
#     "sidecar>=0.8.1",
# ]
# ///

# Visual Cloud-Optimized GeoTIFFs in Lonboard

Lonboard now has the ability to render arbitrary [Cloud-Optimized GeoTIFF] images. This example notebook will go through the process of visualizing a Sentinel-2 True Color image in Lonboard.

[Cloud-Optimized GeoTIFF]: https://cogeo.org/

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run examples/raster-cog-rgb-server.ipynb
```

(The `uvx` command is included when installing `uv`).

## Imports

First our imports. We'll use [Obstore] for accessing S3 and [Async-GeoTIFF] for efficient reading of GeoTIFF files. We use [pillow] (imported as `PIL`) for encoding image tiles to PNG.

[Obstore]: https://developmentseed.org/obstore/latest/
[Async-GeoTIFF]: https://developmentseed.org/async-geotiff/latest/
[pillow]: https://pillow.readthedocs.io/en/stable/

In [ ]:
import io

import numpy as np
from async_geotiff import GeoTIFF
from async_geotiff import Tile as GeoTIFFTile
from async_geotiff.utils import reshape_as_image
from obstore.store import S3Store
from PIL import Image
from sidecar import Sidecar

from lonboard import Map, RasterLayer
from lonboard.raster import EncodedImage

## Access GeoTIFF from AWS S3 bucket with Obstore and Async-GeoTIFF

First, we create an Obstore [`S3Store`] mounted to the bucket of interest. Then we open a [`GeoTIFF`] from a path within that bucket.

[`S3Store`]: https://developmentseed.org/obstore/latest/api/store/aws/#obstore.store.S3Store
[`GeoTIFF`]: https://developmentseed.org/async-geotiff/latest/api/geotiff/#geotiff

In [ ]:
# Obstore store
store = S3Store("sentinel-cogs", region="us-west-2", skip_signature=True)

In [ ]:
# Open our GeoTIFF instance
cog_path = "sentinel-s2-l2a-cogs/18/T/WL/2026/1/S2B_18TWL_20260101_0_L2A/TCI.tif"
geotiff = await GeoTIFF.open(cog_path, store=store)

## Create Render Callback

Lonboard's COG support works by asynchronously fetching COG image tiles _through Python_ and then transferring the tile to JavaScript for visualization.

In this initial version of our support, we require the user to create a "render callback" function that transforms the loaded COG tile to a PNG-formatted RGB image.

The benefit of this approach is that you can use _any Python code_ to perform the rendering characteristics you desire.

In [ ]:
def render_rgb_tile(tile: GeoTIFFTile) -> EncodedImage:
    """Convert the array data from the GeoTIFF to an RGB PNG."""
    # Reshape from (bands, height, width) to (height, width, bands)
    masked_array = reshape_as_image(tile.array.as_masked())

    # Handle nodata regions, making those parts of the image transparent
    rgb = masked_array.data
    mask = masked_array.mask
    if mask.ndim == 0:
        # All pixels valid
        alpha = np.full(rgb.shape[:2], 255, dtype=np.uint8)
    elif mask.ndim == 2:
        alpha = (~mask).astype(np.uint8) * 255
    else:
        alpha = (~mask.any(axis=-1)).astype(np.uint8) * 255

    # Concatenate alpha axis to become (height, width, 4)
    rgba = np.concatenate([rgb, alpha[..., None]], axis=-1)

    # Encode as PNG
    img = Image.fromarray(rgba)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return EncodedImage(data=buf.getvalue(), media_type="image/png")

Now, pass the `geotiff` instance and our "render callback" into the `from_geotiff` constructor:

In [ ]:
layer = RasterLayer.from_geotiff(geotiff, render_tile=render_rgb_tile)

Now we can create a map and display it just like any other layer

In [ ]:
# In JupyterLab, split the screen to render the map on the right
sidecar = Sidecar(anchor="split-right")

In [ ]:
# Create the map
m = Map(layer, height=800)

In [ ]:
# Render the map in the split screen
with sidecar:
    display(m)